# Demonstration of Tool Calling

What did we say?

1. User sends input
2. LLM processes input and either
   1. sends output back
   2. or calls a tool (producing output that we can execute, calling a Python function)
3. THe output of the tool is sent back to the LLM
4. THe LLM can either call another tool or send final output to user
   

Below we are just defining tool calling

In [1]:
# Setup
import os
import json
from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [3]:
# the code to call the LLM!
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Explain tool calling in 2 sentences to my students."}
    ]
)
print(response.content[0].text)

Tool calling lets AI models like ChatGPT connect to external functions or APIs to perform specific tasks they couldn't do on their own, like checking the weather, doing calculations, or searching databases. Think of it as giving the AI the ability to use specialized tools – instead of just talking about math, it can actually call a calculator tool to get precise answers.


In [10]:
tools = [
    {
        "name": "write_file",
        "description": "Write a file to the local filesystem",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "The path to the file to write to"
                },
                "content": {
                    "type": "string",
                    "description": "The content to write to the file"
                }
            },
            "required": ["file_path", "content"]
        }
    },
]

def llm_with_tools(input_user: str) -> str:
    messages = [
        {"role": "user", "content": input_user}
    ]
    
    response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=1024,
    messages=messages,
    tools=tools
)
    return response

def write_file_tool(file_path: str, content: str) -> str:
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)
        return f"File written successfully to {file_path}."
    except Exception as e:
        return f"Error writing file: {str(e)}"

def execute_tool_call(tool_call_output):
    """
    Executes tool calls found in the output Message object. 
    Expects tool_call_output to be a Message object with content containing ToolUseBlock(s).
    Returns the tool results as a list.
    """
    results = []
    for block in tool_call_output.content:
        if getattr(block, "type", None) == "tool_use":
            tool_name = getattr(block, "name", None)
            tool_input = getattr(block, "input", {})
            if tool_name == "write_file":
                # Extract parameters safely
                file_path = tool_input.get("file_path")
                content = tool_input.get("content")
                result = write_file_tool(file_path, content)
                results.append(result)
            else:
                results.append(f"Unknown tool '{tool_name}' requested.")
    return results


output = llm_with_tools("Write a file called test.txt with the content 'Hello, world!'")
print(output)

Message(id='msg_01F4i9R7rUnbYTNUscLufPZu', content=[ToolUseBlock(id='toolu_019STE4oMWT9Yg56k9j8bAFQ', input={'file_path': 'test.txt', 'content': 'Hello, world!'}, name='write_file', type='tool_use', caller={'type': 'direct'})], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=616, output_tokens=76, server_tool_use=None, service_tier='standard', inference_geo='not_available'))


In [11]:
print(output.content[0].name)
print(output.content[0].input)

write_file
{'file_path': 'test.txt', 'content': 'Hello, world!'}


In [12]:
# This is exeucting the tool call output that was produced with the LLM using the Anthropic API
execute_tool_call(output)

['File written successfully to test.txt.']